# OHLC图像深度学习训练数据集

这个notebook用于构建深度学习训练数据集，支持：
- 多时间框架图像堆叠（3min, 15min, 1h）
- 可选择收益时间窗口标签
- 分批加载大文件，内存高效
- PyTorch DataLoader兼容

**输出格式**:
- 图像张量: (N, 3, 64, 60) - 3个时间框架堆叠
- 标签张量: (N,) - 单一收益时间窗口
- 支持时间范围过滤和数据对齐验证

In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple, Union
from torch.utils.data import Dataset, DataLoader, random_split
import gc
import warnings
from tqdm import tqdm
import glob
warnings.filterwarnings('ignore')

print("导入完成")
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

导入完成
PyTorch版本: 2.7.1+cu118
CUDA可用: True


In [2]:
# 配置参数
DATA_DIR = Path('/home/craz/crypto/model_training/ohlc_img_dataset')

# 时间框架和收益时间窗口映射
TIMEFRAMES = ['3min', '15min', '1h']
RETURN_HORIZON_MAP = {
    '1h': '1h', '2h': '2h', '4h': '4h', '8h': '8h', 
    '12h': '12h', '16h': '16h', '18h': '18h',
    '1d': '1d', '1d6h': '1d6h', '1d12h': '1d12h', 
    '2d': '2d', '3d': '3d', '4d': '4d', '5d': '5d', '6d': '6d', '7d': '7d'
}

# 设备配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {DEVICE}")

# 内存管理配置
MAX_MEMORY_GB = 8  # 最大内存使用量(GB)
BATCH_LOAD_SIZE = 10000  # 每批加载的样本数

print(f"数据目录: {DATA_DIR}")
print(f"可用收益时间窗口: {list(RETURN_HORIZON_MAP.keys())}")

使用设备: cuda
数据目录: /home/craz/crypto/model_training/ohlc_img_dataset
可用收益时间窗口: ['1h', '2h', '4h', '8h', '12h', '16h', '18h', '1d', '1d6h', '1d12h', '2d', '3d', '4d', '5d', '6d', '7d']


In [3]:
def find_available_files(data_dir: Path) -> Dict[str, Dict[str, List[Path]]]:
    """扫描可用的数据文件"""
    files_info = {'images': {tf: [] for tf in TIMEFRAMES}, 'labels': {}}
    
    print("📁 扫描可用数据文件...")
    
    # 扫描图像文件
    for timeframe in TIMEFRAMES:
        pattern = f"images_{timeframe}_*.pt"
        pt_files = list(data_dir.glob(pattern))
        pattern = f"images_{timeframe}_*.npz"
        npz_files = list(data_dir.glob(pattern))
        
        files_info['images'][timeframe] = sorted(pt_files + npz_files)
        print(f"  {timeframe}: {len(files_info['images'][timeframe])} 个图像文件")
    
    # 扫描标签文件
    for horizon_key, horizon_str in RETURN_HORIZON_MAP.items():
        pattern = f"labels_{horizon_str}_*.pt"
        pt_files = list(data_dir.glob(pattern))
        pattern = f"labels_{horizon_str}_*.npz"
        npz_files = list(data_dir.glob(pattern))
        
        files_info['labels'][horizon_key] = sorted(pt_files + npz_files)
        if len(files_info['labels'][horizon_key]) > 0:
            print(f"  {horizon_key}: {len(files_info['labels'][horizon_key])} 个标签文件")
    
    # 移除空的标签类别
    files_info['labels'] = {k: v for k, v in files_info['labels'].items() if len(v) > 0}
    
    return files_info

def extract_date_from_filename(filename: str) -> Optional[datetime]:
    """从文件名提取日期"""
    try:
        # 提取 YYYY-MM 格式的日期
        import re
        match = re.search(r'(\d{4})-(\d{2})', filename)
        if match:
            year, month = int(match.group(1)), int(match.group(2))
            return datetime(year, month, 1)
    except:
        pass
    return None

def filter_files_by_date_range(files: List[Path], start_date: Optional[str], end_date: Optional[str]) -> List[Path]:
    """根据日期范围过滤文件"""
    if not start_date and not end_date:
        return files
    
    start_dt = datetime.strptime(start_date, '%Y-%m') if start_date else None
    end_dt = datetime.strptime(end_date, '%Y-%m') if end_date else None
    
    filtered_files = []
    for file_path in files:
        file_date = extract_date_from_filename(file_path.name)
        if file_date:
            if start_dt and file_date < start_dt:
                continue
            if end_dt and file_date > end_dt:
                continue
            filtered_files.append(file_path)
    
    return sorted(filtered_files)

def load_single_file(file_path: Path) -> Dict:
    """加载单个文件（支持.pt和.npz格式）"""
    try:
        if file_path.suffix == '.pt':
            return torch.load(file_path, weights_only=False)
        elif file_path.suffix == '.npz':
            npz_data = np.load(file_path, allow_pickle=True)
            # 转换npz格式为字典
            result = {}
            for key in npz_data.files:
                data = npz_data[key]
                if key in ['images', 'labels'] and isinstance(data, np.ndarray):
                    result[key] = torch.from_numpy(data)
                else:
                    result[key] = data.item() if data.ndim == 0 else data
            return result
    except Exception as e:
        print(f"❌ 加载文件失败 {file_path}: {e}")
        return None

print("工具函数定义完成")

工具函数定义完成


In [4]:
class StackedOHLCDataset(Dataset):
    """堆叠多时间框架OHLC图像数据集"""
    
    def __init__(self, 
                 data_dir: Path,
                 label_horizon: str = '1d',
                 start_date: Optional[str] = None,
                 end_date: Optional[str] = None,
                 batch_load_size: int = 10000,
                 preload_all: bool = False):
        """
        初始化数据集
        
        Args:
            data_dir: 数据文件目录
            label_horizon: 收益时间窗口 ('1h', '1d', '7d' 等)
            start_date: 开始日期 (格式: 'YYYY-MM')
            end_date: 结束日期 (格式: 'YYYY-MM')
            batch_load_size: 批量加载大小
            preload_all: 是否预加载所有数据到内存
        """
        self.data_dir = Path(data_dir)
        self.label_horizon = label_horizon
        self.batch_load_size = batch_load_size
        self.preload_all = preload_all
        
        # 验证参数
        if label_horizon not in RETURN_HORIZON_MAP:
            raise ValueError(f"不支持的收益时间窗口: {label_horizon}. 可选: {list(RETURN_HORIZON_MAP.keys())}")
        
        # 扫描文件
        self.files_info = find_available_files(self.data_dir)
        
        # 按日期范围过滤文件
        self._filter_files_by_date(start_date, end_date)
        
        # 验证数据对齐
        self._validate_data_alignment()
        
        # 构建索引映射
        self._build_index_mapping()
        
        # 预加载数据（如果需要）
        if self.preload_all:
            self._preload_all_data()
        else:
            self.loaded_batches = {}  # 缓存已加载的批次
    
    def _filter_files_by_date(self, start_date: Optional[str], end_date: Optional[str]):
        """根据日期范围过滤文件"""
        print(f"\n📅 过滤日期范围: {start_date or '开始'} 到 {end_date or '结束'}")
        
        # 过滤图像文件
        for timeframe in TIMEFRAMES:
            original_count = len(self.files_info['images'][timeframe])
            self.files_info['images'][timeframe] = filter_files_by_date_range(
                self.files_info['images'][timeframe], start_date, end_date
            )
            filtered_count = len(self.files_info['images'][timeframe])
            print(f"  {timeframe}: {original_count} -> {filtered_count} 个文件")
        
        # 过滤标签文件
        if self.label_horizon in self.files_info['labels']:
            original_count = len(self.files_info['labels'][self.label_horizon])
            self.files_info['labels'][self.label_horizon] = filter_files_by_date_range(
                self.files_info['labels'][self.label_horizon], start_date, end_date
            )
            filtered_count = len(self.files_info['labels'][self.label_horizon])
            print(f"  标签({self.label_horizon}): {original_count} -> {filtered_count} 个文件")
    
    def _validate_data_alignment(self):
        """验证数据对齐"""
        print(f"\n🔍 验证数据对齐...")
        
        # 检查是否有标签文件
        if self.label_horizon not in self.files_info['labels']:
            raise ValueError(f"找不到收益时间窗口 '{self.label_horizon}' 的标签文件")
        
        # 检查各时间框架文件数量是否一致
        file_counts = [len(self.files_info['images'][tf]) for tf in TIMEFRAMES]
        label_count = len(self.files_info['labels'][self.label_horizon])
        
        if len(set(file_counts + [label_count])) != 1:
            print(f"⚠️  文件数量不一致:")
            for tf in TIMEFRAMES:
                print(f"    {tf}: {len(self.files_info['images'][tf])} 个文件")
            print(f"    标签({self.label_horizon}): {label_count} 个文件")
            raise ValueError("数据文件数量不对齐")
        
        print(f"✅ 数据对齐验证通过: {file_counts[0]} 个月份的数据")
        
        # 验证文件名对应关系
        self._validate_filename_alignment()
    
    def _validate_filename_alignment(self):
        """验证文件名对应关系"""
        # 提取所有图像文件的日期标识符
        image_dates = set()
        for tf in TIMEFRAMES:
            for file_path in self.files_info['images'][tf]:
                date_str = file_path.name.split('_')[-1].replace('.pt', '').replace('.npz', '')
                image_dates.add(date_str)
        
        # 提取标签文件的日期标识符
        label_dates = set()
        for file_path in self.files_info['labels'][self.label_horizon]:
            date_str = file_path.name.split('_')[-1].replace('.pt', '').replace('.npz', '')
            label_dates.add(date_str)
        
        # 检查对应关系
        if image_dates != label_dates:
            missing_in_labels = image_dates - label_dates
            missing_in_images = label_dates - image_dates
            if missing_in_labels:
                print(f"⚠️  标签文件缺失的日期: {missing_in_labels}")
            if missing_in_images:
                print(f"⚠️  图像文件缺失的日期: {missing_in_images}")
            raise ValueError("图像和标签文件日期不对应")
        
        print(f"✅ 文件名对应关系验证通过")
    
    def _build_index_mapping(self):
        """构建索引映射"""
        print(f"\n📊 构建数据索引映射...")
        
        self.file_groups = []  # 每个组包含同一月份的所有文件
        self.cumulative_sizes = []  # 累积样本数量
        self.total_samples = 0
        
        # 按文件名排序确保对齐
        sorted_files = {}
        for tf in TIMEFRAMES:
            sorted_files[tf] = sorted(self.files_info['images'][tf], key=lambda x: x.name)
        sorted_label_files = sorted(self.files_info['labels'][self.label_horizon], key=lambda x: x.name)
        
        # 构建文件组
        for i in range(len(sorted_label_files)):
            file_group = {
                'images': {tf: sorted_files[tf][i] for tf in TIMEFRAMES},
                'labels': sorted_label_files[i],
                'samples': None  # 延迟加载时确定
            }
            self.file_groups.append(file_group)
        
        # 快速扫描文件获取样本数量
        print("正在扫描文件大小...")
        for i, group in enumerate(tqdm(self.file_groups)):
            # 从标签文件获取样本数（通常最小）
            label_data = load_single_file(group['labels'])
            if label_data and 'labels' in label_data:
                if isinstance(label_data['labels'], torch.Tensor):
                    samples = len(label_data['labels'])
                else:
                    samples = len(label_data['labels'])
                group['samples'] = samples
                self.total_samples += samples
                self.cumulative_sizes.append(self.total_samples)
            else:
                raise ValueError(f"无法从标签文件获取样本数: {group['labels']}")
        
        print(f"✅ 索引映射完成: {len(self.file_groups)} 个文件组, 总计 {self.total_samples:,} 个样本")
    
    def _preload_all_data(self):
        """预加载所有数据到内存"""
        print(f"\n🔄 预加载所有数据到内存...")
        
        all_images = []
        all_labels = []
        
        for group in tqdm(self.file_groups, desc="加载文件"):
            # 加载图像文件
            images_data = {}
            for tf in TIMEFRAMES:
                img_data = load_single_file(group['images'][tf])
                if img_data and 'images' in img_data:
                    images_data[tf] = img_data['images']
                else:
                    raise ValueError(f"加载图像文件失败: {group['images'][tf]}")
            
            # 堆叠图像 (N, 3, 64, 60)
            stacked_images = torch.stack([
                images_data['3min'],
                images_data['15min'], 
                images_data['1h']
            ], dim=1).squeeze(2)  # 去除原有的channel维度
            
            # 加载标签文件
            label_data = load_single_file(group['labels'])
            if label_data and 'labels' in label_data:
                labels = label_data['labels']
            else:
                raise ValueError(f"加载标签文件失败: {group['labels']}")
            
            all_images.append(stacked_images)
            all_labels.append(labels)
        
        # 合并所有数据
        self.all_images = torch.cat(all_images, dim=0)
        self.all_labels = torch.cat(all_labels, dim=0)
        
        print(f"✅ 预加载完成: 图像 {self.all_images.shape}, 标签 {self.all_labels.shape}")
        print(f"   内存使用: {self.all_images.numel() * 4 / 1024**3:.2f} GB (图像) + {self.all_labels.numel() * 4 / 1024**2:.2f} MB (标签)")
    
    def _find_file_group_for_index(self, idx: int) -> Tuple[int, int]:
        """查找索引对应的文件组和组内索引"""
        for group_idx, cumulative_size in enumerate(self.cumulative_sizes):
            if idx < cumulative_size:
                start_idx = self.cumulative_sizes[group_idx - 1] if group_idx > 0 else 0
                local_idx = idx - start_idx
                return group_idx, local_idx
        raise IndexError(f"索引 {idx} 超出范围")
    
    def _load_file_group(self, group_idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """加载指定文件组的数据"""
        if group_idx in self.loaded_batches:
            return self.loaded_batches[group_idx]
        
        group = self.file_groups[group_idx]
        
        # 加载图像文件
        images_data = {}
        for tf in TIMEFRAMES:
            img_data = load_single_file(group['images'][tf])
            if img_data and 'images' in img_data:
                images_data[tf] = img_data['images']
            else:
                raise ValueError(f"加载图像文件失败: {group['images'][tf]}")
        
        # 堆叠图像 (N, 3, 64, 60)
        stacked_images = torch.stack([
            images_data['3min'],
            images_data['15min'], 
            images_data['1h']
        ], dim=1).squeeze(2)  # 去除原有的channel维度
        
        # 加载标签文件
        label_data = load_single_file(group['labels'])
        if label_data and 'labels' in label_data:
            labels = label_data['labels']
        else:
            raise ValueError(f"加载标签文件失败: {group['labels']}")
        
        # 缓存数据（内存管理）
        if len(self.loaded_batches) > 5:  # 最多缓存5个批次
            # 移除最旧的缓存
            oldest_key = min(self.loaded_batches.keys())
            del self.loaded_batches[oldest_key]
            gc.collect()
        
        self.loaded_batches[group_idx] = (stacked_images, labels)
        
        return stacked_images, labels
    
    def __len__(self) -> int:
        return self.total_samples
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.preload_all:
            return self.all_images[idx], self.all_labels[idx]
        else:
            # 动态加载
            group_idx, local_idx = self._find_file_group_for_index(idx)
            stacked_images, labels = self._load_file_group(group_idx)
            return stacked_images[local_idx], labels[local_idx]
    
    def get_data_info(self) -> Dict:
        """获取数据集信息"""
        return {
            'total_samples': self.total_samples,
            'num_files': len(self.file_groups),
            'label_horizon': self.label_horizon,
            'timeframes': TIMEFRAMES,
            'image_shape': '(N, 3, 64, 60)',
            'label_shape': '(N,)',
            'preloaded': self.preload_all
        }

print("StackedOHLCDataset 类定义完成")

StackedOHLCDataset 类定义完成


In [5]:
def create_training_dataset(data_dir: str,
                          label_horizon: str = '1d',
                          start_date: Optional[str] = None,
                          end_date: Optional[str] = None,
                          preload_all: bool = False) -> Tuple[StackedOHLCDataset, DataLoader]:
    """
    创建训练数据集和数据加载器
    
    Args:
        data_dir: 数据文件目录
        label_horizon: 收益时间窗口 ('1h', '1d', '7d' 等)
        start_date: 开始日期 (格式: 'YYYY-MM')
        end_date: 结束日期 (格式: 'YYYY-MM')  
        preload_all: 是否预加载所有数据到内存
    
    Returns:
        dataset, dataloader
    """
    print(f"🚀 创建训练数据集...")
    print(f"参数配置:")
    print(f"  - 数据目录: {data_dir}")
    print(f"  - 收益时间窗口: {label_horizon}")
    print(f"  - 时间范围: {start_date or '开始'} ~ {end_date or '结束'}")
    print(f"  - 预加载模式: {'是' if preload_all else '否'}")
    
    # 创建数据集
    dataset = StackedOHLCDataset(
        data_dir=Path(data_dir),
        label_horizon=label_horizon,
        start_date=start_date,
        end_date=end_date,
        preload_all=preload_all
    )
    
    # 显示数据集信息
    info = dataset.get_data_info()
    print(f"\n📊 数据集创建完成:")
    print(f"  - 总样本数: {info['total_samples']:,}")
    print(f"  - 文件数量: {info['num_files']}")
    print(f"  - 图像形状: {info['image_shape']}")
    print(f"  - 标签形状: {info['label_shape']}")
    
    return dataset

In [6]:
import time

# 测试配置（使用小数据集）
TEST_CONFIG = {
    'data_dir': DATA_DIR,
    'label_horizon': '1d',
    'start_date': '2025-03',
    'end_date': '2025-06'
}

# 测试动态加载模式
print("\n🔄 测试动态加载模式...")
try:
    start_time = time.time()
    
    dataset_dynamic = create_training_dataset(
        **TEST_CONFIG, preload_all=False
    )
    
except Exception as e:
    print(f"  ❌ 动态加载模式测试失败: {e}")


🔄 测试动态加载模式...
🚀 创建训练数据集...
参数配置:
  - 数据目录: /home/craz/crypto/model_training/ohlc_img_dataset
  - 收益时间窗口: 1d
  - 时间范围: 2025-03 ~ 2025-06
  - 预加载模式: 否
📁 扫描可用数据文件...
  3min: 66 个图像文件
  15min: 66 个图像文件
  1h: 66 个图像文件
  1h: 66 个标签文件
  2h: 66 个标签文件
  4h: 66 个标签文件
  8h: 66 个标签文件
  12h: 66 个标签文件
  16h: 66 个标签文件
  18h: 66 个标签文件
  1d: 66 个标签文件
  1d6h: 66 个标签文件
  1d12h: 66 个标签文件
  2d: 66 个标签文件
  3d: 66 个标签文件
  4d: 66 个标签文件
  5d: 66 个标签文件
  6d: 66 个标签文件
  7d: 66 个标签文件

📅 过滤日期范围: 2025-03 到 2025-06
  3min: 66 -> 4 个文件
  15min: 66 -> 4 个文件
  1h: 66 -> 4 个文件
  标签(1d): 66 -> 4 个文件

🔍 验证数据对齐...
✅ 数据对齐验证通过: 4 个月份的数据
✅ 文件名对应关系验证通过

📊 构建数据索引映射...
正在扫描文件大小...


100%|██████████| 4/4 [00:00<00:00, 22.99it/s]

✅ 索引映射完成: 4 个文件组, 总计 318,641 个样本

📊 数据集创建完成:
  - 总样本数: 318,641
  - 文件数量: 4
  - 图像形状: (N, 3, 64, 60)
  - 标签形状: (N,)


In [7]:
train_ratio = 0.7

n_total = len(dataset_dynamic)
n_train  = int(n_total * train_ratio)
n_val = n_total - n_train

train_ds, val_ds = random_split(
    dataset_dynamic,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)   # 固定随机种子，结果可复现
)


train_dataloader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dataloader  = DataLoader(val_ds,  batch_size=256, shuffle=False)


# print(f"Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}")
# for xb, yb in train_dataloader:
#     print("X batch:", xb.shape, "y batch:", yb.shape)
#     break

In [8]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=(5,3), stride=(3,1), dilation=(2,1), padding=(12,1)),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),
            nn.MaxPool2d((2, 1), stride=(2, 1)),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(5,3), stride=(3,1), dilation=(2,1), padding=(12,1)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),
            nn.MaxPool2d((2, 1), stride=(2, 1)),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=(5,3), stride=(3,1), dilation=(2,1), padding=(12,1)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),
            nn.MaxPool2d((2, 1), stride=(2, 1)),
        )
        self.fc1 = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(46080, 1),
        )
        self.softmax = nn.Softmax(dim=1)
       
    def forward(self, x):
        # x = x.reshape(-1,1,64,60)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x.reshape(-1,46080)
        x = self.fc1(x)
        # x = self.softmax(x)
        return x

In [9]:
use_gpu = torch.cuda.is_available()
device = torch.device("cuda" if use_gpu else "cpu")

net = Net()
if use_gpu:
    net = net.to(device)
    if torch.cuda.device_count() > 1:
        net = nn.DataParallel(net)

def train_loop(dataloader, net, loss_fn, optimizer):
    """训练循环 - 修复版本"""
    running_loss = 0.0
    current = 0
    net.train()
    
    with tqdm(dataloader) as t:
        for batch, (X, y) in enumerate(t):
            X = X.to(device)
            y = y.to(device)
            
            # 前向传播
            y_pred = net(X)
            
            # 确保标签类型正确（回归任务用float）
            if loss_fn.__class__.__name__ in ['L1Loss', 'MSELoss']:
                y = y.float()
            else:
                y = y.long()
            
            loss = loss_fn(y_pred.squeeze(), y)  # 确保维度匹配
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 正确的损失计算
            batch_loss = loss.item()
            running_loss = (len(X) * batch_loss + running_loss * current) / (len(X) + current)
            current += len(X)
            t.set_postfix({'running_loss': running_loss})
    
    return running_loss

def val_loop(dataloader, net, loss_fn):
    """验证循环 - 修复版本"""
    running_loss = 0.0
    current = 0
    net.eval()
    
    with torch.no_grad():
        with tqdm(dataloader) as t:
            for batch, (X, y) in enumerate(t):
                X = X.to(device)
                y = y.to(device)
                
                # 前向传播
                y_pred = net(X)
                
                # 确保标签类型正确（回归任务用float）
                if loss_fn.__class__.__name__ in ['L1Loss', 'MSELoss']:
                    y = y.float()
                else:
                    y = y.long()
                
                loss = loss_fn(y_pred.squeeze(), y)  # 确保维度匹配

                # 修复：正确的损失计算（与训练一致）
                batch_loss = loss.item()
                running_loss = (len(X) * batch_loss + running_loss * current) / (len(X) + current)
                current += len(X)
                t.set_postfix({'running_loss': running_loss})
            
    return running_loss

In [10]:
import os
import time
import datetime
import torch
import torch.nn as nn

def train_model(
    net, 
    train_dataloader, 
    val_dataloader, 
    loss_fn, 
    optimizer, 
    epochs=100, 
    early_stopping_epoch=5,
    start_epoch=0, 
    save_dir="./model_checkpoint"
):
    # 记录训练和验证损失
    train_losses = []
    val_losses = []

    # 记录最佳验证损失
    min_val_loss = float("inf")
    last_min_ind = -1

    # 模型存储目录
    start_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = os.path.join(save_dir, start_time)
    os.makedirs(save_dir, exist_ok=True)

    # 训练循环
    for epoch in range(start_epoch, epochs):
        print(f"\n===== Epoch {epoch} =====")
        
        # 训练
        train_loss = train_loop(train_dataloader, net, loss_fn, optimizer)
        train_losses.append(train_loss)

        # 验证
        val_loss = val_loop(val_dataloader, net, loss_fn)
        val_losses.append(val_loss)

        # 保存当前模型快照
        ckpt_name = f"baseline_epoch_{epoch}_train_{train_loss:.5f}_val_{val_loss:.5f}.pt"
        ckpt_path = os.path.join(save_dir, ckpt_name)
        torch.save(net.state_dict(), ckpt_path)
        print(f"✔ Saved checkpoint: {ckpt_path}")

        # 早停逻辑
        if val_loss < min_val_loss:
            min_val_loss = val_loss
            last_min_ind = epoch
            best_ckpt_path = ckpt_path
        elif epoch - last_min_ind >= early_stopping_epoch:
            print(f"⏹ Early stopping at epoch {epoch} (no improvement for {early_stopping_epoch} epochs)")
            break

    print("\n🎉 Training done!")
    print(f"Best epoch: {last_min_ind}, best val_loss: {min_val_loss:.5f}")

    return best_ckpt_path

best_model_path = train_model(
    net=net,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    loss_fn=nn.MSELoss(),
    optimizer=torch.optim.Adam(net.parameters(), lr=5e-5),
    epochs=100,
    early_stopping_epoch=5,
    start_epoch=0,
    save_dir="./model_checkpoint"
)

print(f"Best model saved at: {best_model_path}")


===== Epoch 0 =====


100%|██████████| 374/374 [00:09<00:00, 37.70it/s, running_loss=0.00947]


✔ Saved checkpoint: ./model_checkpoint/20250915_203810/baseline_epoch_0_train_0.15097_val_0.00947.pt

===== Epoch 1 =====


100%|██████████| 374/374 [00:12<00:00, 29.15it/s, running_loss=0.00533]


✔ Saved checkpoint: ./model_checkpoint/20250915_203810/baseline_epoch_1_train_0.03752_val_0.00533.pt

===== Epoch 2 =====


...

✔ Saved checkpoint: ./model_checkpoint/20250915_203810/baseline_epoch_20_train_0.00454_val_0.00527.pt

===== Epoch 21 =====


100%|██████████| 374/374 [00:09<00:00, 38.17it/s, running_loss=0.00461]


✔ Saved checkpoint: ./model_checkpoint/20250915_203810/baseline_epoch_21_train_0.00443_val_0.00461.pt

===== Epoch 22 =====


100%|██████████| 374/374 [00:09<00:00, 37.97it/s, running_loss=0.00406]


✔ Saved checkpoint: ./model_checkpoint/20250915_203810/baseline_epoch_22_train_0.00435_val_0.00406.pt
⏹ Early stopping at epoch 22 (no improvement for 5 epochs)

🎉 Training done!
Best epoch: 17, best val_loss: 0.00397
Best model saved at: ./model_checkpoint/20250915_203810/baseline_epoch_17_train_0.00479_val_0.00397.pt


## 训练数据集使用指南

### 快速开始

```python
# 创建训练数据集
dataset, dataloader = create_training_dataset(
    data_dir='/home/craz/crypto/model_training/ohlc_img_dataset',
    label_horizon='1d',      # 预测1天收益率
    start_date='2021-01',    # 开始月份
    end_date='2021-12',      # 结束月份
    batch_size=32,           # 批次大小
    shuffle=True,            # 打乱数据
    preload_all=False        # 动态加载模式
)

# 训练循环示例
for epoch in range(num_epochs):
    for batch_idx, (images, labels) in enumerate(dataloader):
        # images: (batch_size, 3, 64, 60) - 堆叠的3个时间框架图像
        # labels: (batch_size,) - 收益率标签
        
        # 移动到GPU (如果可用)
        images = images.to(device)
        labels = labels.to(device)
        
        # 前向传播和反向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### 主要特性

1. **多时间框架堆叠**: 自动将3min、15min、1h的图像堆叠为 (N, 3, 64, 60)
2. **灵活收益预测**: 支持1h到7d的多种收益时间窗口
3. **时间范围过滤**: 可指定训练的起始和结束月份
4. **内存管理**: 支持预加载和动态加载两种模式
5. **数据对齐验证**: 确保图像和标签完美对应

### 参数说明

| 参数 | 类型 | 说明 | 示例 |
|------|------|------|------|
| `data_dir` | str | 数据文件目录 | '/path/to/dataset' |
| `label_horizon` | str | 收益时间窗口 | '1h', '1d', '7d' |
| `start_date` | str | 开始日期 | '2021-01' |
| `end_date` | str | 结束日期 | '2021-12' |
| `batch_size` | int | 批次大小 | 32 |
| `shuffle` | bool | 是否打乱 | True |
| `preload_all` | bool | 预加载模式 | False |

### 内存管理策略

**预加载模式** (`preload_all=True`):
- 优点: 训练速度快，无IO延迟
- 缺点: 占用大量内存
- 适用: 小数据集或内存充足的情况

**动态加载模式** (`preload_all=False`):
- 优点: 内存占用小，支持大数据集
- 缺点: 有IO延迟，速度稍慢
- 适用: 大数据集或内存受限的情况

### 数据格式

**输入图像**: (batch_size, 3, 64, 60)
- 通道0: 3分钟时间框架OHLC图像
- 通道1: 15分钟时间框架OHLC图像  
- 通道2: 1小时时间框架OHLC图像

**输出标签**: (batch_size,)
- 浮点数，表示未来收益率 (小数形式，如0.05表示5%)

### 常见使用场景

1. **短期预测**: `label_horizon='1h'` 或 `'4h'`
2. **中期预测**: `label_horizon='1d'` 或 `'3d'` 
3. **长期预测**: `label_horizon='7d'`
4. **回测验证**: 设置特定的`start_date`和`end_date`

### 注意事项

- 确保数据文件已通过`generate_dataset_complete.ipynb`生成
- 检查可用的收益时间窗口，不是所有窗口都有对应的数据文件
- 大数据集建议使用动态加载模式以避免内存不足
- 建议先用小数据集测试模型架构和训练逻辑